# wrangle / household / networth

**Intent**
- Outputs `household_networth_current_dollars`, net worth at the latest month end.
- Outputs `household_networth_savings_rate_pct`, the savings rate over the period.
- Gives constants, with the starting wealth the portfolio models need.
- Closes the loop between the plan's forecasts and what happens.

**Data**
- Personal data is git-ignored in `data/` and `project/`, with synthetic placeholders committed.
- `training` is twelve synthetic months of net worth, income and spending.
- It is synthetic until the `bank` table exists.
- The `Balances` tab of the `Bank` workbook holds a reading per account over time from 2021-05.

**Status**
- The real series is not loaded and nothing is released.
- It carries no evaluation, so the golden replay and the Train cell assertions are the gate.

**Future**
- Overall: the work still to do before the first release:
  - Add a `bank` table to `wrangle`.
  - Build the net worth series, reconcile it to income less spending, and separate saving from market return.
  - Add the tracking fan against the base, low and high paths once `portfolio/return` is real.
  - Add property, super and other assets with dated values, and liabilities with their rates.

## Setup ([6.7.1](https://github.com/ggear/asystem/blob/master/src/rue/mlmodels/src/build/resources/doc/model.md#671-setup))

In [ ]:
from workbench import setup_workspace

ws = setup_workspace("wrangle/household/networth", {"kind": "local"})

INPUT = ws.inputs[0]

## Data ([6.7.2](https://github.com/ggear/asystem/blob/master/src/rue/mlmodels/src/build/resources/doc/model.md#672-data))

In [ ]:
from workbench import data_load

training_pl, training_path = data_load(
    ws,
    "training",
    "household_networth_training",
    run=1,
    source="builder",
    build=lambda: {
        "month": list(range(12)),
        "net_worth_dollars": [1000000.0 + 5000.0 * month for month in range(12)],
        "income_dollars": [12000.0] * 12,
        "spending_dollars": [9000.0] * 12,
    },
)
print("training", training_pl.height)

## Train ([6.7.4](https://github.com/ggear/asystem/blob/master/src/rue/mlmodels/src/build/resources/doc/model.md#674-train))

In [ ]:
income = float(training_pl["income_dollars"].sum())
spending = float(training_pl["spending_dollars"].sum())
CONSTANTS = dict(zip(ws.outputs, [round(float(training_pl["net_worth_dollars"][-1]), 0), round((income - spending) / income * 100.0, 2)], strict=True))
print("constants", CONSTANTS)
assert list(CONSTANTS.values()) == [1055000.0, 25.0], "the summary did not give the latest net worth and the savings rate"

## Package ([6.7.6](https://github.com/ggear/asystem/blob/master/src/rue/mlmodels/src/build/resources/doc/model.md#676-package))

In [ ]:
from workbench import package_workspace

golden = {INPUT: [1.0, None], "expect": ["ok", "error"]}
package_workspace(
    ws,
    "static",
    golden,
    trained_on="synthetic monthly net worth, income and spending, training split",
    inputs=[training_path],
    notes="latest net worth and the savings rate over the period",
    constants=CONSTANTS,
)